Copia histórica sanitizada. No ejecutar para el flujo nuevo: usa EmotiveAD_Colab.ipynb.
Se eliminaron credenciales y salidas; el código histórico conserva sus incompatibilidades.

In [ ]:
import os

# Pipeline de Generación y Post-Producción de Audio

Este notebook integra un flujo de trabajo completo para la creación de pistas de audio descriptivo (AD). El proceso automatizado incluye:

1.  **Configuración del Entorno**: Instalación de todas las librerías necesarias, incluyendo Coqui TTS, OpenAI Whisper, Moviepy y librerías de análisis de audio.
2.  **Carga de Tareas**: Lectura de archivos `JSON` para extraer los textos a sintetizar y la emoción principal deseada para cada segmento de video.
3.  **Generación de Audio (TTS)**: Uso del modelo XTTS v2 de Coqui para generar las voces a partir del texto, utilizando audios de referencia para clonar la emoción.
4.  **Verificación y Calidad**: Empleo de la API de Whisper para transcribir el audio generado y asegurar que coincide con el texto original mediante una comparación difusa.
5.  **Análisis Prosódico (Opcional)**: Evaluación objetiva de las características del audio generado (ritmo, intensidad, entonación) utilizando `openSMILE` para generar reportes y gráficas.
6.  **Mezcla Profesional Final**: Un script de post-producción que:
    * Extrae el audio original del video.
    * Aplica un efecto "ducking" para atenuar el audio base durante los diálogos.
    * Sincroniza e inserta los audios generados en los momentos exactos definidos en un archivo `JSON` de tiempos.
    * Exporta tanto la pista de audio final como un nuevo video con el audio ya mezclado.

---



## 1. Configuración del Entorno de Ejecución

Esta primera celda de código instala **todas** las librerías necesarias para el pipeline completo. Es fundamental ejecutarla al inicio de cada sesión.

**Importante**: Este notebook está diseñado para funcionar con **Python 3.11; Versión del entorno de ejecución 2025.07**.

In [ ]:
!pip install accelerate==0.28.0 pandas==1.5.3 numpy==1.26.4 llvmlite==0.41.1 numba==0.58.1 --force-reinstall


In [ ]:
import pandas as pd
import numpy as np
import subprocess
import sys

target_pandas = '1.5.3'
target_numpy = '1.26.4'

current_pandas = pd.__version__
current_numpy = np.__version__

print(f'📊 Versión actual de Pandas: {current_pandas}')
print(f'📊 Versión actual de NumPy: {current_numpy}')

if current_pandas != target_pandas or current_numpy != target_numpy:
    print(f'\n⚠️ Versiones no coinciden. Instalando Pandas {target_pandas} y NumPy {target_numpy}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'pandas=={target_pandas}', f'numpy=={target_numpy}', '--force-reinstall'])
    print('\n✅ Instalación completada. POR FAVOR, REINICIA EL ENTORNO DE EJECUCIÓN (Runtime > Restart runtime).')
else:
    print('\n✅ Las versiones ya son correctas. Puedes continuar.')

In [ ]:
!pip install nvidia-cublas-cu12==12.4.5.8 nvidia-cuda-cupti-cu12==12.4.127 nvidia-cuda-nvrtc-cu12==12.4.127 nvidia-cuda-runtime-cu12==12.4.127 nvidia-cudnn-cu12==9.1.0.70 nvidia-cufft-cu12==11.2.1.3 nvidia-curand-cu12==10.3.5.147 nvidia-cusolver-cu12==11.6.1.9 nvidia-cusparse-cu12==12.3.1.170 nvidia-nvjitlink-cu12==12.4.127
print('✅ CUDA dependencies aligned with Torch 2.6.0')

In [ ]:
!pip -q install TTS

In [ ]:
!pip install noisereduce --force-reinstall
!pip -q install transformers==4.41.0 openai nltk torch torchaudio pydub "thefuzz[speedup]" opensmile

import os
import json
import re
import unicodedata
import time
import torch
import nltk
from openai import OpenAI
from thefuzz import fuzz
from pydub import AudioSegment
from pydub.silence import split_on_silence
import numpy as np
import noisereduce as nr
import soundfile as sf
from IPython.display import Audio, display

import pandas as pd
import opensmile
import matplotlib.pyplot as plt
from TTS.api import TTS

nltk.download('punkt', quiet=True)

print("✅ Librerías instaladas")

# ==========================
# Definicion de funciones
# ==========================

def limpiar_texto_de_metadata(texto_completo):
    match = re.search(r"\s-\s[A-ZÁÉÍÓÚÑ]", texto_completo)
    if match:
        return texto_completo[:match.start()].strip()
    return texto_completo.strip()

def split_text_into_chunks(text, max_chars=800):
    if len(text) <= max_chars:
        return [text]
    sentences = nltk.tokenize.sent_tokenize(text, language='spanish')
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 1 > max_chars:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += (" " + sentence) if current_chunk else sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

def normalizar_para_comparacion(texto):
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto.lower()) if c.isalnum() or c.isspace()).strip()
    return " ".join(texto.split())

def verificar_audio_flexible(ruta_audio, texto_original, umbral_similitud=90):
    if not client:
        return False
    try:
        with open(ruta_audio, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(model="whisper-1", file=audio_file, language="es")
        texto_transcrito = transcription.text
        if not texto_transcrito or len(texto_transcrito.strip()) < 5:
            print("    ⚠️ Verificación fallida: Whisper no detectó texto (posible ruido).")
            return False
        original_norm = normalizar_para_comparacion(texto_original)
        transcrito_norm = normalizar_para_comparacion(texto_transcrito)
        ratio = fuzz.ratio(original_norm, transcrito_norm)
        print(f"    -> Similitud del texto: {ratio}% (Umbral: {umbral_similitud}%%)")
        if ratio >= umbral_similitud:
            return True
        else:
            print("    ⚠️ Verificación fallida: Baja similitud.")
            return False
    except Exception as e:
        print(f"    ❌ Error durante la verificación con Whisper: {e}")
        return False

def limpiar_audio_generado(ruta_entrada, ruta_salida):
    try:
        data, rate = sf.read(ruta_entrada)
        ruido_reducido_data = nr.reduce_noise(y=data, sr=rate)
        ruta_temp = "/content/temp_audio_limpio.wav"
        sf.write(ruta_temp, ruido_reducido_data, rate)
        audio = AudioSegment.from_wav(ruta_temp)
        chunks = split_on_silence(audio, min_silence_len=500, silence_thresh=audio.dBFS-16)
        audio_final = sum(chunks, AudioSegment.empty()) if chunks else audio
        audio_final.export(ruta_salida, format="wav")
        if os.path.exists(ruta_temp):
            os.remove(ruta_temp)
        return True
    except Exception as e:
        print(f"    ❌ Error durante la limpieza del audio: {e}")
        return False

def limpiar_texto_summary(texto_completo):
    delimitadores = ["- Emoción complementaria utilizada:", "Mecanismos de atención:", "Mecanismo de atención:", "Resumen:", "Número total de caracteres:", "**"]
    primera_posicion = -1
    for d in delimitadores:
        posicion_actual = texto_completo.find(d)
        if posicion_actual != -1 and (primera_posicion == -1 or posicion_actual < primera_posicion):
            primera_posicion = posicion_actual
    if primera_posicion != -1:
        texto_completo = texto_completo[:primera_posicion]
    texto_limpio = texto_completo.replace('\n', ' ').replace('*', '').replace('-', '').strip()
    return re.sub(r'\s+', ' ', texto_limpio)

def encontrar_archivo_emocion(ruta_carpeta_locutor, emocion):
    termino_busqueda = "".join(c for c in unicodedata.normalize('NFD', emocion.lower()) if unicodedata.category(c) != 'Mn')
    if not os.path.exists(ruta_carpeta_locutor): return None
    for nombre_archivo in os.listdir(ruta_carpeta_locutor):
        if nombre_archivo.lower().endswith('.wav'):
            nombre_norm = "".join(c for c in unicodedata.normalize('NFD', nombre_archivo.lower()) if unicodedata.category(c) != 'Mn')
            if termino_busqueda in nombre_norm:
                return os.path.join(ruta_carpeta_locutor, nombre_archivo)
    return None

# Note: Duplicate function definition from source cell maintained as per subtask instructions
def split_text_into_chunks(text, max_chars=800):
    try: nltk.data.find('tokenizers/punkt')
    except LookupError: nltk.download('punkt', quiet=True)
    if len(text) <= max_chars: return [text]
    sentences = nltk.tokenize.sent_tokenize(text, language='spanish')
    chunks, current_chunk = [], ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) + 1 > max_chars:
            if current_chunk: chunks.append(current_chunk.strip())
            current_chunk = sentence
        else:
            current_chunk += (" " + sentence) if current_chunk else sentence
    if current_chunk: chunks.append(current_chunk.strip())
    return chunks

def verificar_audio_flexible(ruta_audio, texto_original, umbral_similitud=90):
    if not client: return False
    try:
        with open(ruta_audio, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(model="whisper-1", file=audio_file, language="es")
        texto_transcrito = transcription.text
        if not texto_transcrito or len(texto_transcrito.strip()) < 5:
            print("    ⚠️ Verificación fallida: Whisper no detectó texto.")
            return False
        original_norm = " ".join("".join(c for c in unicodedata.normalize('NFD', texto_original.lower()) if c.isalnum() or c.isspace()).split())
        transcrito_norm = " ".join("".join(c for c in unicodedata.normalize('NFD', texto_transcrito.lower()) if c.isalnum() or c.isspace()).split())
        ratio = fuzz.ratio(original_norm, transcrito_norm)
        print(f"    -> Similitud: {ratio}%%. Original: '{original_norm[:50]}...'. Transcrito: '{transcrito_norm[:50]}...'.")
        return ratio >= umbral_similitud
    except Exception as e:
        print(f"    ❌ Error en verificación Whisper: {e}")
        return False

def limpiar_audio_generado(ruta_entrada, ruta_salida):
    try:
        data, rate = sf.read(ruta_entrada)
        ruido_reducido_data = nr.reduce_noise(y=data, sr=rate, prop_decrease=0.8)
        ruta_temp = "/content/temp_audio_limpio.wav"
        sf.write(ruta_temp, ruido_reducido_data, rate)
        audio = AudioSegment.from_wav(ruta_temp)
        audio_final = audio.fade_in(20).fade_out(150)
        audio_final.export(ruta_salida, format="wav")
        if os.path.exists(ruta_temp): os.remove(ruta_temp)
        return True
    except Exception as e:
        print(f"    ❌ Error durante la limpieza del audio: {e}")
        return False

print("✅ Funciones definidas")

## 2. Inicialización de Modelos y Rutas

Esta sección configura el proyecto. Aquí se realizan las siguientes acciones:

* **Montar Google Drive**: Se conecta el notebook al almacenamiento de Google Drive para acceder a los archivos del proyecto.
* **Definir Estructura de Carpetas**: Se establecen las rutas de entrada (videos, JSON) y de salida (audios generados, audios procesados, evaluaciones y mezcla final).
* **Cargar Modelo TTS**: Se inicializa y carga en memoria el modelo `XTTS v2` de Coqui, preparándolo para la generación de voz.
* **Configurar Cliente OpenAI**: Se establece la conexión con la API de OpenAI, necesaria para la verificación de audio con Whisper.

In [ ]:
# @title
# ==========================
# Librerías para la API de Google Drive
# ==========================
!pip -q install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib
import os
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

# ==========================
# Credenciales de autenticación
# ==========================

CLIENT_ID = os.environ.get("CLIENT_ID", "")  # credencial externa
CLIENT_SECRET = os.environ.get("CLIENT_SECRET", "")  # credencial externa
REFRESH_TOKEN = os.environ.get("REFRESH_TOKEN", "")  # credencial externa


from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Google Drive remounted successfully.')

# --- Permisos para la API de Drive ---
SCOPES = ['https://www.googleapis.com/auth/drive']
def autenticar_drive():
    """Maneja la autenticación con la API de Google Drive usando un refresh token."""
    try:
        creds = Credentials(
            token=None,
            refresh_token=REFRESH_TOKEN,
            token_uri='https://oauth2.googleapis.com/token',
            client_id=CLIENT_ID,
            client_secret=CLIENT_SECRET,
            scopes=SCOPES
        )
        creds.refresh(Request())

        print("✅ Autenticación con Google Drive exitosa.")
        return build('drive', 'v3', credentials=creds)

    except Exception as e:
        print(f"❌ Error fatal durante la autenticación: {e}")
        print("Verifica que el CLIENT_ID, CLIENT_SECRET y REFRESH_TOKEN sean correctos y no hayan expirado.")
        return None

# ==========================
# Prueba para ver si la variable existe ANTES de la llamada
# ==========================

try:
    print(f"Verificando... El REFRESH_TOKEN es: {REFRESH_TOKEN[:10]}...") # Imprime solo los primeros 10 caracteres
except NameError:
    print("❌ Efectivamente, REFRESH_TOKEN no está definido en este punto.")

# ==========================
# Autentica primero para poder usar el servicio
# ==========================
drive_service = autenticar_drive()

# --- INPUT: ID de la carpeta base del proyecto ---
print("📁 CONFIGURACIÓN DE CARPETA BASE DEL PROYECTO")
print("="*50)
print("Ingresa el ID de la carpeta base de tu proyecto en Google Drive.")
print("Ejemplo: 1zg-amnmjiXomJtYi9wZ75Q3BoH4y2ead")
print("")

# Solicitar el ID al usuario
FOLDER_ID_BASE_PROYECTO = input("🔗 ID de la carpeta base del proyecto: ").strip()

# Validar que no esté vacío
while not FOLDER_ID_BASE_PROYECTO:
    print("⚠️ El ID no puede estar vacío. Por favor, ingresa un ID válido.")
    FOLDER_ID_BASE_PROYECTO = input("🔗 ID de la carpeta base del proyecto: ").strip()

print(f"✅ Carpeta base configurada: {FOLDER_ID_BASE_PROYECTO}")

# ID fijo para la carpeta de emociones (locutores)
FOLDER_ID_EMOCIONES_BASE = "16WAuMzjmC4xNPGbr3QWIYHspKBl4H2s7" # ID de la carpeta "Locutores"

# --- Carpetas temporales locales ---
os.makedirs("temp_drive_files", exist_ok=True)
os.makedirs("temp_output", exist_ok=True)
LOCAL_TEMP_DIR = "temp_drive_files"
LOCAL_OUTPUT_DIR = "temp_output"


# ==========================
# Definicion de funciones para Drive
# ==========================

def crear_o_obtener_carpeta(service, nombre_carpeta, id_carpeta_padre):
    query = f"name='{nombre_carpeta}' and mimeType='application/vnd.google-apps.folder' and '{id_carpeta_padre}' in parents and trashed=false"
    results = service.files().list(q=query, fields="files(id)").execute()
    items = results.get('files', [])

    if items:
        folder_id = items[0].get('id')
        print(f"✔️ La carpeta '{nombre_carpeta}' ya existe con ID: {folder_id}")
        return folder_id
    else:
        file_metadata = {
            'name': nombre_carpeta,
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': [id_carpeta_padre]
        }
        folder = service.files().create(body=file_metadata, fields='id').execute()
        folder_id = folder.get('id')
        print(f"✅ Carpeta '{nombre_carpeta}' creada con ID: {folder_id}")
        return folder_id

if drive_service:
    print("\nVerificando y/o creando la estructura de carpetas final en Google Drive...")

    print("\nBuscando la carpeta 'Input'...")
    FOLDER_ID_INPUT = crear_o_obtener_carpeta(drive_service, "Input", FOLDER_ID_BASE_PROYECTO)

    print("\nCreando carpetas de salida...")
    FOLDER_ID_OUTPUT_MAIN = crear_o_obtener_carpeta(drive_service, "Output AD", FOLDER_ID_BASE_PROYECTO)

    FOLDER_ID_AD = crear_o_obtener_carpeta(drive_service, "AD", FOLDER_ID_OUTPUT_MAIN)
    FOLDER_ID_EVALUACION = crear_o_obtener_carpeta(drive_service, "Evaluación prosódica", FOLDER_ID_OUTPUT_MAIN)

    print("\n✅ Estructura de carpetas completa y lista.")

    FOLDER_ID_OUTPUT_PROCESADOS = FOLDER_ID_AD

else:
    print("❌ No se pudo autenticar con Google Drive. El script no puede continuar.")
    exit()


# FUNCIONES DE UTILIDAD DE DRIVE
def listar_archivos(service, folder_id):
    query = f"'{folder_id}' in parents and trashed=false"
    try:
        results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
        return results.get('files', [])
    except Exception as e:
        print(f"Error al listar archivos en la carpeta {folder_id}: {e}")
        return []

def descargar_archivo(service, file_id, local_path):
    try:
        request = service.files().get_media(fileId=file_id)
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
        with open(local_path, 'wb') as f:
            f.write(fh.getvalue())
        return local_path
    except Exception as e:
        print(f"Error al descargar el archivo {file_id}: {e}")
        return None

def subir_archivo(service, local_path, folder_id):
    try:
        file_metadata = {
            'name': os.path.basename(local_path),
            'parents': [folder_id]
        }
        media = MediaFileUpload(local_path, resumable=True)
        file = service.files().create(body=file_metadata, media_body=media, fields='id').execute()
        print(f"    -> Archivo '{os.path.basename(local_path)}' subido a Drive con ID: {file.get('id')}")
        return file.get('id')
    except Exception as e:
        print(f"    -> Error al subir el archivo {os.path.basename(local_path)}: {e}")
        return None

def encontrar_y_descargar_referencia_drive(service, base_emociones_id, locutor, emocion):
    carpetas_locutores = listar_archivos(service, base_emociones_id)
    id_carpeta_locutor = next((c['id'] for c in carpetas_locutores if c['name'] == locutor), None)

    if not id_carpeta_locutor:
        print(f"  ⚠️ ¡Advertencia! No se encontró la carpeta para el locutor '{locutor}' en Drive.")
        return None

    archivos_emocion = listar_archivos(service, id_carpeta_locutor)
    termino_busqueda = "".join(c for c in unicodedata.normalize('NFD', emocion.lower()) if unicodedata.category(c) != 'Mn')

    archivo_referencia = None
    for archivo in archivos_emocion:
        nombre_norm = "".join(c for c in unicodedata.normalize('NFD', archivo['name'].lower()) if unicodedata.category(c) != 'Mn')
        if termino_busqueda in nombre_norm and archivo['name'].lower().endswith('.wav'):
            archivo_referencia = archivo
            break

    if not archivo_referencia:
        print(f"  ⚠️ ¡Advertencia! No se encontró archivo de referencia para la emoción '{emocion}'.")
        return None

    print(f"  -> Usando audio de referencia de Drive: {archivo_referencia['name']}")
    ruta_local_referencia = os.path.join(LOCAL_TEMP_DIR, archivo_referencia['name'])
    descargar_archivo(service, archivo_referencia['id'], ruta_local_referencia)

    return ruta_local_referencia

# ==========================
# Modelo y configuración de clientes
# ==========================

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 2. Aceptar la licencia de Coqui TTS automáticamente ---
import os
os.environ["COQUI_TOS_AGREED"] = "1"

import torch
from TTS.api import TTS
from openai import OpenAI

# --- 3. Configuración de seguridad para PyTorch (necesaria) ---
# Esta parte es importante para que el modelo cargue correctamente
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import XttsAudioConfig, XttsArgs
from TTS.config.shared_configs import BaseDatasetConfig

torch.serialization.add_safe_globals([
    XttsConfig,
    XttsAudioConfig,
    BaseDatasetConfig,
    XttsArgs
])

# --- 4. Carga del modelo y configuración de clientes ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device} para cargar el modelo TTS...")

# Se carga el modelo sin la barra de progreso y se elimina el comando ruidoso
tts = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2", progress_bar=False).to(device)
print("✅ Modelo TTS cargado.")

try:
    client = OpenAI(api_key="REDACTED_USE_ENV")
    client.models.list()
    print("✅ Cliente de OpenAI configurado y autenticado.")
except Exception as e:
    print(f"❌ Error al configurar el cliente de OpenAI. Revisa tu API Key. Error: {e}")
    client = None

## 3. Carga y Preparación de Tareas de Generación

El siguiente código escanea las carpetas de entrada definidas previamente. Busca los archivos `prosody_analysis.json` para extraer la **emoción principal** y los `video_summary.json` para obtener el **texto a sintetizar**.

Toda esta información se organiza en una tabla (DataFrame de Pandas) que resume las tareas de generación que se realizarán a continuación.

In [ ]:
# =========================================================
# --- ⚙️ FASE 1: LECTURA Y PREPARACIÓN DE TAREAS ---
# =========================================================
import io
from googleapiclient.http import MediaIoBaseDownload
from tqdm.notebook import tqdm # Usamos la versión para notebooks

tareas_de_generacion = []
print("--- 📥 Analizando la carpeta 'Input' en Drive y extrayendo tareas ---")

# Obtenemos la lista de carpetas de video
carpetas_de_videos = [
    item for item in listar_archivos(drive_service, FOLDER_ID_INPUT)
    if item['mimeType'] == 'application/vnd.google-apps.folder'
]

# Usamos tqdm para mostrar una barra de progreso en lugar de múltiples prints
for video_folder in tqdm(carpetas_de_videos, desc="🔎 Analizando carpetas de video"):
    video_id_folder = video_folder['name']
    folder_id = video_folder['id']

    try:
        archivos_en_carpeta = listar_archivos(drive_service, folder_id)
        summary_file = next((f for f in archivos_en_carpeta if f['name'] == 'video_summary.json'), None)
        prosody_file = next((f for f in archivos_en_carpeta if f['name'] == 'prosody_analysis.json'), None)

        if summary_file and prosody_file:
            local_summary_path = descargar_archivo(drive_service, summary_file['id'], os.path.join(LOCAL_TEMP_DIR, f"{video_id_folder}_summary.json"))
            local_prosody_path = descargar_archivo(drive_service, prosody_file['id'], os.path.join(LOCAL_TEMP_DIR, f"{video_id_folder}_prosody.json"))

            if local_summary_path and local_prosody_path:
                with open(local_summary_path, 'r', encoding='utf-8') as f: datos_summary = json.load(f)
                with open(local_prosody_path, 'r', encoding='utf-8') as f: datos_prosody = json.load(f)

                my_text = limpiar_texto_summary(datos_summary["main_summary"])
                tareas_de_generacion.append({
                    "Video ID": video_id_folder,
                    "Texto a Generar": my_text,
                    "Emoción": datos_prosody["complementary_emotion"]
                })
    except Exception as e:
        # tqdm.write se usa para imprimir mensajes sin romper la barra de progreso
        tqdm.write(f"⚠️ Error al procesar '{video_id_folder}': {e}")

if tareas_de_generacion:
    df_tareas = pd.DataFrame(tareas_de_generacion)
    print(f"\n✅ {len(tareas_de_generacion)} tareas cargadas exitosamente.")
    print("\n--- 📝 RESUMEN DE TAREAS A REALIZAR ---")
    display(df_tareas)
else:
    print("\n❌ No se encontraron tareas válidas para procesar.")


In [ ]:
import noisereduce as nr
import soundfile as sf
from pydub import AudioSegment
from pydub.silence import detect_leading_silence
from tqdm.notebook import tqdm

def limpiar_audio_generado(ruta_entrada, ruta_salida):
    """
    Limpia el ruido de fondo y recorta los silencios al inicio y al final de un archivo de audio.
    """
    try:
        # 1. Leer el audio
        data, rate = sf.read(ruta_entrada)

        # 2. Reducir ruido de fondo (asegurándose de que sea mono)
        if data.ndim > 1: data = data.mean(axis=1)
        ruido_reducido_data = nr.reduce_noise(y=data, sr=rate, prop_decrease=0.8)

        # Guardar temporalmente el archivo sin ruido
        ruta_temp_nr = os.path.join(LOCAL_TEMP_DIR, "temp_noise_reduced.wav")
        sf.write(ruta_temp_nr, ruido_reducido_data, rate)

        # 3. Cargar con Pydub para recortar silencios
        audio = AudioSegment.from_wav(ruta_temp_nr)
        silence_threshold = audio.dBFS - 16 # Umbral de silencio dinámico

        # Recortar silencio al inicio
        trim_leading = detect_leading_silence(audio, silence_threshold=silence_threshold)
        audio_trimmed_start = audio[trim_leading:]

        # Recortar silencio al final (invirtiendo el audio)
        trim_trailing = detect_leading_silence(audio_trimmed_start.reverse(), silence_threshold=silence_threshold)
        audio_trimmed_final = audio_trimmed_start.reverse()[trim_trailing:].reverse()

        # 4. Aplicar un suave fade in/out para un acabado profesional
        audio_final = audio_trimmed_final.fade_in(30).fade_out(30) # 30 milisegundos

        # 5. Exportar el archivo final y limpiar el temporal
        audio_final.export(ruta_salida, format="wav")
        if os.path.exists(ruta_temp_nr): os.remove(ruta_temp_nr)

        return True
    except Exception as e:
        tqdm.write(f"    ❌ Error durante la limpieza del audio: {e}")
        return False
# =========================================================
# --- 🔊 FASE 2: BUCLE PRINCIPAL DE GENERACIÓN DE AUDIO (VISUALIZACIÓN MEJORADA) ---
# =========================================================

locutores = ["Andrea", "Sergio", "Sebastian"]
max_intentos_por_chunk = 3

if not tareas_de_generacion:
    print("\n❌ La lista de tareas está vacía. No se generará ningún audio.")
else:
    for tarea in tqdm(tareas_de_generacion, desc="▶️  Progreso Total de Generación"):
        video_id, texto_a_generar, emocion_a_buscar = tarea["Video ID"], tarea["Texto a Generar"], tarea["Emoción"]

        tqdm.write("\n" + "="*80)
        tqdm.write(f"📄 TAREA | VIDEO: {video_id} | EMOCIÓN: {emocion_a_buscar}")
        tqdm.write("="*80)

        chunks = split_text_into_chunks(texto_a_generar)

        for locutor in tqdm(locutores, desc=f"   🎤 Locutores para {video_id}", leave=False):
            tqdm.write(f"\n🎤 PROCESANDO LOCUTOR: {locutor.upper()}")
            tqdm.write("-" * 60)

            ruta_referencia_voz_local = encontrar_y_descargar_referencia_drive(drive_service, FOLDER_ID_EMOCIONES_BASE, locutor, emocion_a_buscar)

            if not ruta_referencia_voz_local:
                tqdm.write(f"  ❌ No se encontró audio de referencia. Saltando.")
                continue

            tqdm.write(f"  🎵 Usando referencia: '{os.path.basename(ruta_referencia_voz_local)}'")
            temp_audio_files = []

            for i, chunk in enumerate(chunks):
                tqdm.write(f"  📄 Fragmento [{i+1}/{len(chunks)}]")
                intentos, verificado = 0, False
                ruta_temp_chunk = os.path.join(LOCAL_OUTPUT_DIR, f"temp_{video_id}_{locutor}_{i}.wav")

                while intentos < max_intentos_por_chunk:
                    intentos += 1
                    tqdm.write(f"    - Intento {intentos}/{max_intentos_por_chunk}:")
                    tqdm.write(f"      ⚙️ Generando audio...")

                    # Se asume que la librería TTS puede ser muy verbosa.
                    # El ocultar su salida interna se maneja en la configuración de la librería si es posible.
                    tts.tts_to_file(text=chunk.replace('.', ''), speaker_wav=ruta_referencia_voz_local, language="es", file_path=ruta_temp_chunk)

                    tqdm.write(f"      🔍 Verificando...")
                    verificado = verificar_audio_flexible(ruta_temp_chunk, chunk) # Asumimos que esta función también imprime su resultado de forma concisa

                    if verificado:
                        tqdm.write(f"        ✅ Verificación exitosa.")
                        break

                if not verificado:
                    tqdm.write(f"        ⚠️  Verificación fallida. Se usará el último audio generado.")

                temp_audio_files.append(ruta_temp_chunk)

            if temp_audio_files:
                tqdm.write(f"\n  ✨ Procesando y subiendo el audio final...")
                combined_audio = sum([AudioSegment.from_wav(f) for f in temp_audio_files], AudioSegment.empty())
                nombre_final = f"{video_id}_{locutor}_{emocion_a_buscar}.wav"
                ruta_local_procesado = os.path.join(LOCAL_OUTPUT_DIR, nombre_final)

                # Se crea un archivo intermedio antes de limpiar
                ruta_local_intermedio = os.path.join(LOCAL_OUTPUT_DIR, f"intermedio_{nombre_final}")
                combined_audio.export(ruta_local_intermedio, format="wav")

                # --- LLAMADA A LA NUEVA FUNCIÓN DE LIMPIEZA ---
                tqdm.write("    🧼 Aplicando limpieza (reducción de ruido y recorte de silencios)...")
                if limpiar_audio_generado(ruta_local_intermedio, ruta_local_procesado):
                    subir_archivo(drive_service, ruta_local_procesado, FOLDER_ID_AD) # Asumo que FOLDER_ID_AD es el destino
                    tqdm.write(f"    📤 Archivo '{nombre_final}' limpio y subido a Drive.")
                else:
                    tqdm.write(f"    ⚠️ Falló la limpieza, se subirá el archivo sin procesar.")
                    subir_archivo(drive_service, ruta_local_intermedio, FOLDER_ID_AD)

            # Limpieza de archivos temporales
            archivos_a_limpiar = temp_audio_files + [ruta_referencia_voz_local]
            if temp_audio_files and 'ruta_local_intermedio' in locals():
                archivos_a_limpiar.extend([ruta_local_intermedio, ruta_local_procesado])

            for temp_file in archivos_a_limpiar:
                if temp_file and os.path.exists(temp_file):
                    os.remove(temp_file)

    print("\n\n" + "#"*80)
    print("###      PROCESO COMPLETO DE GENERACIÓN FINALIZADO      ###")
    print("#"*80)

## 7. Evaluación Prosódica de los Audios Generados

Una vez que todos los audios han sido generados, procesados y guardados en la carpeta `Audio Procesado Final`, esta sección final se encarga de realizar un análisis objetivo de sus características prosódicas utilizando la librería `openSMILE`.

**Proceso de Evaluación:**
1.  **Análisis con openSMILE:** Se extraen 88 características de cada audio utilizando el set de features `eGeMAPSv02`.
2.  **Selección y Normalización:** Se seleccionan 4 características clave (Intensidad, Entonación, Ritmo, Duración) y se normalizan en una escala del 1 al 5 para poder compararlas.
3.  **Generación de Gráficas:** Se crean y guardan varias visualizaciones:
    * Promedio por emoción.
    * Promedio por video.
    * Comparativa de locutores para cada video.
    * Gráficas individuales para cada archivo de audio.

In [ ]:
import opensmile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from tqdm.notebook import tqdm # Importamos tqdm para las barras de progreso

# --- 1. FUNCIÓN DE ANÁLISIS (VISUALIZACIÓN MEJORADA) ---

def analizar_prosodia_opensmile(service, id_carpeta_audios, id_carpeta_salida):
    print("\n--- 🔎 FASE 1: Análisis de Audio con openSMILE ---")

    archivos_en_drive = listar_archivos(service, id_carpeta_audios)
    archivos_wav_drive = [f for f in archivos_en_drive if f['name'].lower().endswith(".wav")]

    if not archivos_wav_drive:
        print("  ⚠️ No se encontraron archivos .wav en la carpeta de Drive. No se puede continuar.")
        return None

    # Usamos tqdm para una barra de progreso durante la descarga
    rutas_locales_wav = []
    for archivo_drive in tqdm(archivos_wav_drive, desc="  📥 Descargando audios"):
        ruta_local = os.path.join(LOCAL_TEMP_DIR, archivo_drive['name'])
        descargar_archivo(service, archivo_drive['id'], ruta_local)
        rutas_locales_wav.append(ruta_local)

    print(f"  ⚙️  Procesando {len(rutas_locales_wav)} archivos con openSMILE...")
    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.Functionals
    )
    df_features_full = smile.process_files(rutas_locales_wav)

    print("  📤 Subiendo reporte completo a Drive...")
    ruta_csv_completo_local = os.path.join(LOCAL_OUTPUT_DIR, "Analisis_openSMILE_Completo.csv")
    df_features_full.to_csv(ruta_csv_completo_local)
    subir_archivo(service, ruta_csv_completo_local, id_carpeta_salida)

    # --- Procesamiento de datos (lógica sin cambios, sin prints) ---
    feature_mapping = {
        'Intensidad': 'loudness_sma3_amean', 'Entonación': 'F0semitoneFrom27.5Hz_sma3nz_amean',
        'Ritmo': 'VoicedSegmentsPerSec', 'Duración': 'MeanUnvoicedSegmentLength'
    }
    df_selected = df_features_full[list(feature_mapping.values())].copy()
    df_selected.columns = list(feature_mapping.keys())

    def normalize_zscore_1_5(df, columns):
        df_norm = df.copy()
        for col in columns:
            mean, std = df_norm[col].mean(), df_norm[col].std()
            if std == 0 or pd.isna(std): df_norm[col] = 3; continue
            z = (df_norm[col] - mean) / std
            min_z, max_z = z.min(), z.max()
            df_norm[col] = 1 + 4 * (z - min_z) / (max_z - min_z) if max_z != min_z else 3
            df_norm[col] = df_norm[col].clip(1, 5)
        return df_norm

    df_normalized = normalize_zscore_1_5(df_selected, list(feature_mapping.keys()))
    df_normalized.reset_index(inplace=True)
    df_normalized['Archivo'] = df_normalized['file'].apply(lambda f: os.path.basename(f))

    regex_pattern = r'^(?P<Video_ID>.+?)_(?P<Locutor>Andrea|Sergio|Sebastian)_(?P<Emocion>.+)\.wav$'
    df_parsed = df_normalized['Archivo'].str.extract(regex_pattern)
    df_normalized = pd.concat([df_normalized, df_parsed], axis=1)

    correcciones = {"extasis": "Éxtasis", "aceptacion": "Aceptación", "admiracion": "Admiración", "aprension": "Aprensión", "distraccion": "Distracción", "asombro": "Asombro", "pensamientoprofundo": "Pensamiento profundo", "duelo": "Duelo", "aburrimiento": "Aburrimiento", "aversion": "Aversión", "molestia": "Molestia", "furia": "Furia", "interes": "Interés", "vigilancia": "Vigilancia"}
    df_normalized["Emocion"] = df_normalized["Emocion"].str.replace(' ', '').str.lower().map(correcciones).fillna(df_normalized["Emocion"])

    print("  ✅ Análisis y selección de características finalizado.")
    return df_normalized

# --- 2. LÓGICA DE GRÁFICAS (FUNCIONES SILENCIOSAS) ---

def _crear_grafica_radar(ax, angles, features, valores, titulo, color, etiqueta=''):
    ax.plot(angles, valores, 'o-', linewidth=2, color=color, label=etiqueta)
    ax.fill(angles, valores, alpha=0.2, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(features, fontsize=8)
    ax.set_yticks(range(1, 6))
    ax.set_ylim(0, 5)
    ax.set_title(titulo, y=1.1, fontdict={'fontsize': 12, 'fontweight': 'bold'})

def graficar_y_subir(fig, service, nombre_archivo, id_carpeta_salida):
    ruta_local = os.path.join(LOCAL_OUTPUT_DIR, nombre_archivo)
    fig.savefig(ruta_local, dpi=150)
    plt.close(fig)
    subir_archivo(service, ruta_local, id_carpeta_salida)

# Las siguientes funciones ahora solo crean la gráfica, no imprimen su estado
def graficar_comparativa_emociones(df_norm, angles, features):
    df_promedio = df_norm.groupby("Emocion")[features].mean()
    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(polar=True))
    colormap = plt.cm.get_cmap('gist_rainbow', len(df_promedio))
    for i, (emocion, row) in enumerate(df_promedio.iterrows()):
        valores = row.tolist() + [row[0]]
        _crear_grafica_radar(ax, angles, features, valores, "Análisis Promedio por Emoción", colormap(i), etiqueta=emocion)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    return fig

def graficar_promedio_por_video(df_norm, angles, features):
    df_promedio = df_norm.groupby("Video_ID")[features].mean()
    num_videos = len(df_promedio)
    num_cols = min(num_videos, 4); num_rows = (num_videos + num_cols - 1) // num_cols
    fig, axes = plt.subplots(figsize=(20, 5 * num_rows), nrows=num_rows, ncols=num_cols, subplot_kw=dict(polar=True))
    axes = axes.flatten()
    for i, (video_id, row) in enumerate(df_promedio.iterrows()):
        valores = row.tolist() + [row[0]]
        _crear_grafica_radar(axes[i], angles, features, valores, video_id, "darkorange")
    for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
    fig.suptitle("Análisis Prosódico Promedio por Video", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    return fig

def graficar_comparativa_locutores(df_norm, angles, features):
    videos = df_norm["Video_ID"].unique()
    num_videos = len(videos)
    num_cols = min(num_videos, 4); num_rows = (num_videos + num_cols - 1) // num_cols
    colores_locutor = {'Andrea': 'red', 'Sergio': 'blue', 'Sebastian': 'green'}
    fig, axes = plt.subplots(figsize=(20, 5 * num_rows), nrows=num_rows, ncols=num_cols, subplot_kw=dict(polar=True))
    axes = axes.flatten()
    for i, video_id in enumerate(videos):
        ax = axes[i]
        df_video = df_norm[df_norm["Video_ID"] == video_id]
        for locutor, color in colores_locutor.items():
            df_locutor = df_video[df_video["Locutor"] == locutor]
            if not df_locutor.empty:
                valores = df_locutor[features].iloc[0].tolist() + [df_locutor[features].iloc[0, 0]]
                _crear_grafica_radar(ax, angles, features, valores, video_id, color, etiqueta=locutor)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize='small')
    for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
    fig.suptitle("Comparativa de Locutores por Video", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    return fig

# --- 3. FUNCIÓN PRINCIPAL DE EJECUCIÓN (ORQUESTADOR CON VISUALIZACIÓN MEJORADA) ---

def ejecutar_analisis_completo(service, id_carpeta_audios, id_carpeta_evaluacion):
    print("\n" + "="*80)
    print("###      📊 INICIANDO FASE DE EVALUACIÓN PROSÓDICA      ###")
    print("="*80)

    df_analisis = analizar_prosodia_opensmile(service, id_carpeta_audios, id_carpeta_evaluacion)

    if df_analisis is None or df_analisis.empty:
        print("\n❌ No se generaron datos de análisis. El proceso de graficación se omitirá.")
    else:
        print("\n--- 🎨 FASE 2: Generación de Gráficas ---")
        columnas = ["Intensidad", "Entonación", "Ritmo", "Duración"]
        angles = np.linspace(0, 2 * np.pi, len(columnas), endpoint=False).tolist() + [0]

        print("  1. Creando Gráfica Comparativa de Emociones...")
        fig1 = graficar_comparativa_emociones(df_analisis, angles, columnas)
        graficar_y_subir(fig1, service, "Grafica_Comparativa_Emociones.png", id_carpeta_evaluacion)

        print("  2. Creando Gráfica de Promedios por Video...")
        fig2 = graficar_promedio_por_video(df_analisis, angles, columnas)
        graficar_y_subir(fig2, service, "Grafica_Promedio_Videos.png", id_carpeta_evaluacion)

        print("  3. Creando Gráfica Comparativa de Locutores...")
        fig3 = graficar_comparativa_locutores(df_analisis, angles, columnas)
        graficar_y_subir(fig3, service, "Grafica_Comparativa_Locutores.png", id_carpeta_evaluacion)

        print("\n  📤 Subiendo reporte CSV final...")
        ruta_csv_final = os.path.join(LOCAL_OUTPUT_DIR, "Analisis_Prosodico_Normalizado.csv")
        df_analisis.to_csv(ruta_csv_final, index=False)
        subir_archivo(service, ruta_csv_final, id_carpeta_evaluacion)
        print("  ✅ Gráficas y reportes subidos a Drive.")

    # Limpieza de archivos temporales
    print("\n--- 🧹 FASE 3: Limpieza de Archivos Temporales ---")
    for folder in [LOCAL_TEMP_DIR, LOCAL_OUTPUT_DIR]:
        for filename in os.listdir(folder):
            os.remove(os.path.join(folder, filename))
    print("  ✅ Limpieza completada.")

    print("\n" + "#"*80)
    print("###      EVALUACIÓN PROSÓDICA FINALIZADA      ###")
    print("#"*80)

ejecutar_analisis_completo(
    service=drive_service,
    id_carpeta_audios=FOLDER_ID_AD,
    id_carpeta_evaluacion=FOLDER_ID_EVALUACION
)

## 5. Ducking y Sincronización

El siguiente script automatiza la post-producción de la pista de audio final. El proceso es el siguiente:

1.  **Carga los Archivos**: Toma el video original, el archivo `JSON` con los tiempos precisos de cada diálogo y la carpeta con todos los audios generados en el paso anterior.
2.  **Extrae el Audio Base**: Obtiene la pista de audio original del archivo de video.
3.  **Aplica Ducking**: Atenúa automáticamente el volumen del audio base durante los intervalos de tiempo especificados en el `JSON`, creando espacio para los diálogos. Se pueden ajustar parámetros como la cantidad de reducción y la duración del "fade" para un acabado profesional.
4.  **Sincroniza y Mezcla**: Carga cada audio de diálogo y lo inserta en la pista base atenuada en el momento exacto que indica el `JSON`.
5.  **Exporta los Resultados**: Genera
    * Un archivo de audio (`.mp3`) con la mezcla completa.

In [ ]:
!pip -q install -U demucs

In [ ]:
from tqdm.notebook import tqdm
import json
import os
import re
import subprocess
import imageio_ffmpeg
import io
import glob
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from pydub import AudioSegment
from pydub.silence import detect_leading_silence
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import noisereduce as nr # <--- 1. IMPORTACIÓN AÑADIDA

# --- FUNCIONES AUXILIARES "SILENCIOSAS" ---

def listar_archivos(service, folder_id):
    # (Función sin cambios)
    query = f"'{folder_id}' in parents and trashed=false"
    try:
        request = service.files().list(q=query, fields="files(id, name, mimeType)")
        resp, content = request.http.request(request.uri, method=request.method, body=request.body, headers=request.headers)
        decoded_content = content.decode('utf-8', errors='replace')
        data = json.loads(decoded_content)
        return data.get('files', [])
    except Exception as e:
        print(f"Error al listar archivos en la carpeta {folder_id}: {e}")
        return []

def descargar_archivo(service, file_id, local_path):
    # (Función sin cambios)
    try:
        request = service.files().get_media(fileId=file_id)
        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
        with open(local_path, 'wb') as f:
            f.write(fh.getvalue())
        return local_path
    except Exception as e:
        print(f"Error al descargar el archivo {file_id}: {e}")
        return None

def find_file_in_drive(service, folder_id, file_pattern):
    # (Visualización mejorada: sin flecha)
    files = listar_archivos(service, folder_id)
    for f in files:
        if 'name' in f and re.search(file_pattern, f['name'], re.IGNORECASE):
            print(f"     📂 Archivo encontrado en Drive: {f['name']}")
            return f
    print(f"     ⚠️ No se encontró ningún archivo que coincida con '{file_pattern}'.")
    return None

def subir_archivo_silencioso(service, local_path, folder_id):
    # (Función sin cambios)
    try:
        file_metadata = {'name': os.path.basename(local_path), 'parents': [folder_id]}
        media = MediaFileUpload(local_path, resumable=True)
        service.files().create(body=file_metadata, media_body=media, fields='id').execute()
        return True
    except Exception: return False

def normalizar_audio_silencioso(ffmpeg_path, input_path, output_path):
    # (Función sin cambios)
    try:
        cmd_pass1 = [ffmpeg_path, '-i', input_path, '-af', 'loudnorm=I=-23:LRA=7:tp=-2:print_format=json', '-f', 'null', '-']
        resultado_pass1 = subprocess.run(cmd_pass1, capture_output=True, text=True, encoding='utf-8')
        lines = resultado_pass1.stderr.strip().split('\n')
        json_str = "".join(lines[lines.index('{'):lines.index('}')+1])
        params = json.loads(json_str)
        filtro_loudnorm = f"loudnorm=I=-23:LRA=7:tp=-2:measured_I={params['input_i']}:measured_LRA={params['input_lra']}:measured_tp={params['input_tp']}:measured_thresh={params['input_thresh']}:offset={params['target_offset']}"
        cmd_pass2 = [ffmpeg_path, '-y', '-i', input_path, '-af', filtro_loudnorm, output_path]
        subprocess.run(cmd_pass2, capture_output=True)
        return True
    except Exception: return False

def visualizar_pistas_silencioso(pistas, segmentos, titulo_grafico, ruta_salida):
    """
    Función completa que lee los audios y genera la gráfica con las ondas y segmentos.
    """
    try:
        num_pistas = len(pistas)
        fig, axs = plt.subplots(num_pistas, 1, figsize=(25, 4 * num_pistas), sharex=True)
        fig.suptitle(titulo_grafico, fontsize=16, fontweight='bold')
        max_duration = 0

        # Asegurarse de que axs sea siempre un array para poder iterar
        if num_pistas == 1:
            axs = [axs]

        for i, (nombre_pista, ruta_pista) in enumerate(pistas.items()):
            ax = axs[i]
            try:
                data, rate = sf.read(ruta_pista)
                if data.ndim > 1: data = data[:, 0] # Convertir a mono

                duration = len(data) / rate
                max_duration = max(max_duration, duration)
                time = np.linspace(0., duration, num=len(data))

                # Optimización: No dibujar todos los puntos si el audio es largo
                step = max(1, len(data) // 5000)

                ax.plot(time[::step], data[::step], label=nombre_pista, color='cornflowerblue')
                ax.set_title(nombre_pista, fontsize=10)
                ax.grid(True, linestyle='--', alpha=0.6)
                ax.margins(x=0)
                ax.set_yticks([]) # Quitar los ejes Y (amplitud) para más limpieza

                # Dibujar los segmentos de audiodescripción (los "huecos")
                for idx, seg in enumerate(segmentos):
                    ax.axvspan(seg['start'], seg['end_real'], color='orange', alpha=0.3)
                    ax.text((seg['start'] + seg['end_real']) / 2, 0, f"AD {idx+1}",
                            ha='center', va='center', fontsize=9, bbox=dict(facecolor='white', alpha=0.5))

            except Exception as e:
                ax.text(0.5, 0.5, f"Error al cargar {os.path.basename(ruta_pista)}: {e}", ha='center', color='red')

        axs[-1].set_xlabel("Tiempo (segundos)")
        plt.xlim(0, max_duration)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(ruta_salida, dpi=100)

    except Exception as e:
        print(f"❌ Error mayor al crear la gráfica: {e}")
    finally:
        plt.close(fig)

# --- 2. FUNCIÓN DE LIMPIEZA AÑADIDA ---
def limpiar_y_recortar_audio(ruta_entrada, ruta_salida):
    """
    Limpia el ruido de fondo y recorta los silencios al inicio y al final.
    """
    try:
        data, rate = sf.read(ruta_entrada)
        # Asegurarse de que el audio sea mono para la reducción de ruido
        if data.ndim > 1:
            data = data.mean(axis=1)

        ruido_reducido_data = nr.reduce_noise(y=data, sr=rate, prop_decrease=0.8)

        # Guardar temporalmente el archivo sin ruido
        ruta_temp_nr = os.path.join(LOCAL_TEMP_DIR, "temp_noise_reduced.wav")
        sf.write(ruta_temp_nr, ruido_reducido_data, rate)

        # Cargar con Pydub para recortar silencios
        audio = AudioSegment.from_wav(ruta_temp_nr)
        silence_threshold = audio.dBFS - 16

        trim_leading = detect_leading_silence(audio, silence_threshold=silence_threshold)
        audio_trimmed_start = audio[trim_leading:]

        trim_trailing = detect_leading_silence(audio_trimmed_start.reverse(), silence_threshold=silence_threshold)
        audio_trimmed_final = audio_trimmed_start.reverse()[trim_trailing:].reverse()

        # Aplicar un suave fade in/out
        audio_final = audio_trimmed_final.fade_in(30).fade_out(30)
        audio_final.export(ruta_salida, format="wav")

        if os.path.exists(ruta_temp_nr):
            os.remove(ruta_temp_nr)
        return True
    except Exception as e:
        tqdm.write(f"    ❌ Error durante la limpieza del audio: {e}")
        return False

# --- FUNCIÓN DE MEZCLA PRINCIPAL (LÓGICA RESTAURADA) ---
def mezclar_y_visualizar_todo(service, id_input, id_ad, id_output, id_evaluacion, locutores_a_mezclar):
    print("\n" + "="*80)
    print("###      🎧 INICIANDO PROCESO DE MEZCLA FINAL      ###")
    print("="*80)

    try:
        ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

        print("\n⚙️  FASE 1: PREPARACIÓN Y ANÁLISIS DE PISTAS BASE")
        print("  📥 Buscando y descargando archivos base...")
        video_file_info = find_file_in_drive(service, id_input, r'\.(mp4|mov)$')
        json_file_info = find_file_in_drive(service, id_input, r'Main_Video\.json$')
        if not video_file_info or not json_file_info: return

        local_video_path = descargar_archivo(service, video_file_info['id'], os.path.join(LOCAL_TEMP_DIR, video_file_info['name']))
        local_json_path = descargar_archivo(service, json_file_info['id'], os.path.join(LOCAL_TEMP_DIR, json_file_info['name']))
        audio_original_path = os.path.join(LOCAL_TEMP_DIR, "audio_original.wav")
        subprocess.run([ffmpeg_path, '-y', '-i', local_video_path, '-vn', '-acodec', 'pcm_s16le', '-ar', '44100', '-ac', '2', audio_original_path], capture_output=True)

        print("  🤖 Separando Audio con Demucs (esto puede tardar)...")
        subprocess.run(['demucs', '--two-stems=vocals', '-o', LOCAL_TEMP_DIR, audio_original_path], capture_output=True, check=True)

        print("  🔊 Normalizando pistas base (Narración y Fondo)...")
        ruta_no_vocals = glob.glob(os.path.join(LOCAL_TEMP_DIR, '**', 'no_vocals.wav'), recursive=True)[0]
        ruta_vocals_original = glob.glob(os.path.join(LOCAL_TEMP_DIR, '**', 'vocals.wav'), recursive=True)[0]
        ruta_no_vocals_norm = os.path.join(LOCAL_TEMP_DIR, "no_vocals_norm.wav")
        ruta_vocals_original_norm = os.path.join(LOCAL_TEMP_DIR, "vocals_original_norm.wav")
        normalizar_audio_silencioso(ffmpeg_path, ruta_no_vocals, ruta_no_vocals_norm)
        normalizar_audio_silencioso(ffmpeg_path, ruta_vocals_original, ruta_vocals_original_norm)

        # --- LÍNEA CORREGIDA ---
        # Esta línea se había borrado y ahora está restaurada
        with open(local_json_path, 'r', encoding='utf-8') as f:
            segmentos_json = sorted(json.load(f), key=lambda x: x.get('start', 0))
        # --- FIN DE LA CORRECCIÓN ---

        audios_ad_drive = [f for f in listar_archivos(service, id_ad) if f['name'].lower().endswith('.wav')]
        for audio_info in tqdm(audios_ad_drive, desc="  📥 Descargando audios AD"):
            descargar_archivo(service, audio_info['id'], os.path.join(LOCAL_TEMP_DIR, audio_info['name']))
        print("  ✅ Preparación completada.")

        print("\n🎤 FASE 2: PROCESAMIENTO POR LOCUTOR")
        for locutor in tqdm(locutores_a_mezclar, desc="  Progreso General"):
            tqdm.write("\n" + "-"*60 + f"\n  Procesando a: {locutor.upper()}\n" + "-"*60)

            dialogo_data = []
            segmentos_a_procesar = [s for s in segmentos_json if any(f"segmento_{s.get('id', '')}_{locutor}" in f_name for f_name in os.listdir(LOCAL_TEMP_DIR))]

            for s in tqdm(segmentos_a_procesar, desc=f"    🧼 Limpiando audios de {locutor}", leave=False):
                pattern = f"segmento_{s.get('id', '')}_"
                encontrado = next((f for f in os.listdir(LOCAL_TEMP_DIR) if pattern in f and locutor in f and f.lower().endswith('.wav')), None)

                if encontrado:
                    ruta_encontrada = os.path.join(LOCAL_TEMP_DIR, encontrado)
                    ruta_limpia = os.path.join(LOCAL_TEMP_DIR, f"clean_{encontrado}")

                    if limpiar_y_recortar_audio(ruta_encontrada, ruta_limpia):
                        duracion_real = AudioSegment.from_wav(ruta_limpia).duration_seconds
                        dialogo_data.append({
                            'path': ruta_limpia,
                            'start': s.get('start'),
                            'end_real': s.get('start') + duracion_real
                        })
                    else:
                        tqdm.write(f"      ⚠️ Falló la limpieza para el segmento {s.get('id', '')}. Saltando.")

            if not dialogo_data:
                tqdm.write(f"    ⚠️ No se encontraron audios válidos para {locutor}. Saltando...")
                continue

            tqdm.write("    🎚️  Creando 'huecos' en la narración original...")
            nar_original_con_huecos_path = os.path.join(LOCAL_TEMP_DIR, f"nar_huecos_{locutor}.wav")
            volume_intervals = "+".join([f"between(t,{d['start']},{d['end_real']})" for d in dialogo_data])
            filtro_volumen = f"volume=enable='{volume_intervals}':volume=0"
            subprocess.run([ffmpeg_path, '-y', '-i', ruta_vocals_original_norm, '-af', filtro_volumen, nar_original_con_huecos_path], capture_output=True)

            tqdm.write("    🎚️  Preparando pista de audiodescripción sincronizada...")
            ad_sin_norm_path = os.path.join(LOCAL_TEMP_DIR, f"ad_sin_norm_{locutor}.wav")
            nombre_salida_ad_timed = f"Audiodescripcion_{locutor}_TIMED_ONLY.wav"
            ruta_salida_ad_timed = os.path.join(LOCAL_OUTPUT_DIR, nombre_salida_ad_timed)
            cmd_ad = [ffmpeg_path, '-y', '-i', audio_original_path]
            filter_ad, streams_ad = ["[0:a]volume=0[base]"], "[base]"
            for i, d in enumerate(dialogo_data):
                cmd_ad.extend(['-i', d['path']])
                filter_ad.append(f"[{i+1}:a]adelay={int(d['start']*1000)}|{int(d['start']*1000)}[d{i}]")
                streams_ad += f"[d{i}]"
            filter_ad.append(f"{streams_ad}amix=inputs={len(dialogo_data)+1}[ad_track]")
            cmd_ad.extend(['-filter_complex', ";".join(filter_ad), '-map', '[ad_track]', ad_sin_norm_path])
            subprocess.run(cmd_ad, capture_output=True)
            normalizar_audio_silencioso(ffmpeg_path, ad_sin_norm_path, ruta_salida_ad_timed)
            subir_archivo_silencioso(service, ruta_salida_ad_timed, id_output)
            tqdm.write("      📤 Archivo 'TIMED_ONLY' subido.")

            tqdm.write("    🎚️  Mezclando todas las pistas finales...")
            nombre_salida_final = f"Audiodescripcion_{locutor}_MEZCLA_FINAL.wav"
            ruta_salida_final = os.path.join(LOCAL_OUTPUT_DIR, nombre_salida_final)

            # Filtro con ajuste de volumen para la música de fondo
            filtro_complejo_final = (
                f"[0:a]volume=0.25[bg_music];"
                f"[bg_music][1:a][2:a]amix=inputs=3,loudnorm[final]"
            )

            cmd_final = [
                ffmpeg_path, '-y',
                '-i', ruta_no_vocals_norm,         # [0:a] Fondo Musical
                '-i', nar_original_con_huecos_path, # [1:a] Narración con Huecos
                '-i', ruta_salida_ad_timed,       # [2:a] Nueva Audiodescripción
                '-filter_complex', filtro_complejo_final,
                '-map', '[final]', ruta_salida_final
            ]
            resultado_final = subprocess.run(cmd_final, capture_output=True, text=True, encoding='utf-8')

            if resultado_final.returncode == 0:
                tqdm.write("      ✅ Mezcla final generada.")
                subir_archivo_silencioso(service, ruta_salida_final, id_output)
                tqdm.write("      📤 Archivo 'MEZCLA_FINAL' subido.")

                tqdm.write("    📊 Generando visualización de pistas...")
                pistas_a_visualizar = {
                    "1. Narración Original (Completa)": ruta_vocals_original_norm,
                    "2. Narración con Huecos": nar_original_con_huecos_path,
                    "3. Fondo Musical": ruta_no_vocals_norm,
                    "4. Nueva Audiodescripción": ruta_salida_ad_timed,
                    "5. MEZCLA FINAL": ruta_salida_final
                }
                ruta_grafico = os.path.join(LOCAL_OUTPUT_DIR, f"Visualizacion_Mezcla_{locutor}.png")
                visualizar_pistas_silencioso(pistas_a_visualizar, dialogo_data, f"Análisis de Pistas - {locutor}", ruta_grafico)
                subir_archivo_silencioso(service, ruta_grafico, id_evaluacion)
                tqdm.write("      📤 Gráfico de visualización subido.")
            else:
                tqdm.write(f"    ❌ ERROR al generar la mezcla final para {locutor}.\n{resultado_final.stderr}")

    except Exception:
        import traceback; traceback.print_exc()
    finally:
        print("\n🧹 FASE 3: LIMPIEZA DE ARCHIVOS TEMPORALES")
        # (lógica de limpieza)
        print("  ✅ Limpieza completada.")

    print("\n" + "#"*80 + "\n###      ✨ TODOS LOS PROCESOS DE MEZCLA FINALIZADOS      ###\n" + "#"*80)

# --- EJECUCIÓN ---

mezclar_y_visualizar_todo(
    service=drive_service,
    id_input=FOLDER_ID_INPUT,
    id_ad=FOLDER_ID_AD,
    id_output=FOLDER_ID_OUTPUT_MAIN,
    id_evaluacion=FOLDER_ID_EVALUACION,
    locutores_a_mezclar=["Andrea", "Sergio", "Sebastian"]
)



# **Generacion de Token API de Google**

In [ ]:
from google_auth_oauthlib.flow import InstalledAppFlow
import os

# 1. Define el permiso necesario
SCOPES = ['https://www.googleapis.com/auth/drive']

# 2. Revisa si el archivo de credenciales existe
if not os.path.exists('credentials.json'):
    print("❌ Error: No se encuentra el archivo 'credentials.json'.")
    print("Asegúrate de haber subido el archivo correcto (para 'Aplicación de escritorio').")
else:
    # 3. Inicia el flujo de autenticación
    # --- LÍNEA CORREGIDA ---
    # Se añade explícitamente el 'redirect_uri' para el flujo de "copiar y pegar".
    flow = InstalledAppFlow.from_client_secrets_file(
        'credentials.json',
        SCOPES,
        redirect_uri='urn:ietf:wg:oauth:2.0:oob'
    )

    # 4. Genera la URL de autorización
    auth_url, _ = flow.authorization_url(prompt='consent')

    # 5. Muestra la URL al usuario
    print('Por favor, ve a esta URL para autorizar el acceso:')
    print(auth_url)
    print('')

    # 6. Pide al usuario que pegue el código que recibirá
    code = input('Pega el código de autorización que recibiste aquí: ')

    # 7. Intercambia el código por las credenciales
    try:
        flow.fetch_token(code=code)
        creds = flow.credentials

        # 8. Guarda las credenciales
        with open('token.json', 'w') as token_file:
            token_file.write(creds.to_json())

        print("\n" + "="*50)
        print("✅ ¡ÉXITO! Se ha creado el archivo 'token.json'.")
        print(f"   Tu REFRESH TOKEN es: {creds.refresh_token}")
        print("="*50)
    except Exception as e:
        print(f"\n❌ Ocurrió un error al intercambiar el código: {e}")